In [1]:
pip install tensorflow

In [2]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from sklearn.preprocessing import MinMaxScaler

# 1. Datos: [Precio de cierre, Volumen en millones]
# Datos simulados coherentes con la tendencia de Bankinter
datos_bkt = np.array([
    [13.32, 2.1], [13.38, 2.5], [13.37, 1.8], [13.83, 3.2],
    [13.64, 2.0], [13.67, 2.1], [14.40, 4.5], [14.26, 3.0],
    [14.49, 3.5], [14.60, 3.8], [14.90, 4.2], [14.88, 3.9]
])

# 2. Normalización de ambas columnas por separado
scaler = MinMaxScaler()
datos_norm = scaler.fit_transform(datos_bkt)

# 3. Crear ventanas (X incluirá Precio y Volumen, y queremos predecir solo Precio)
def crear_ventanas_multi(datos, ventana=3):
    X, y = [], []
    for i in range(len(datos) - ventana):
        X.append(datos[i:i+ventana, :]) # Todas las columnas (Precio y Vol)
        y.append(datos[i+ventana, 0])    # Solo la columna 0 (Precio)
    return np.array(X), np.array(y)

X, y = crear_ventanas_multi(datos_norm)

# 4. Modelo LSTM
model = Sequential([
    # input_shape ahora es (3 pasos, 2 características)
    LSTM(50, activation='relu', input_shape=(3, 2)),
    Dense(1)
])
model.compile(optimizer='adam', loss='mse')

# 5. Entrenamiento
model.fit(X, y, epochs=600, verbose=0)

# 6. Predicción para mañana
ultima_ventana = datos_norm[-3:].reshape(1, 3, 2)
prediccion_norm = model.predict(ultima_ventana, verbose=0)

# Para des-normalizar el precio, necesitamos un "truco"
# (el scaler espera 2 columnas, así que creamos una fila ficticia)
dummy = np.zeros((1, 2))
dummy[0, 0] = prediccion_norm
prediccion_final = scaler.inverse_transform(dummy)[0, 0]

print(f"Predicción Bankinter (usando Precio+Volumen): {prediccion_final:.3f}€")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Predicción Bankinter (usando Precio+Volumen): 14.975€


/tmp/ipykernel_5147/149702250.py:46: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  dummy[0, 0] = prediccion_norm
